# Acceder a datos CALIPSO de lidar de nubes y aerosoles desde NASA Earthdata


`CAL_LID_L2_01kmCLay-Standard-V5-00` es el producto de datos Cloud-Aerosol Lidar and Infrared Pathfinder Satellite Observation (**CALIPSO**) Lidar Level 2 1 km Cloud Layer. Este producto de datos fue recolectado usando el instrumento Cloud-Aerosol Lidar with Orthogonal Polarization (CALIOP). Dentro de este producto de capa de nubes, generado con una resolución horizontal de 1 km, hay dos clases generales de datos: Column Properties (incluyendo datos de posición y geometría de observación) y Layer Properties. Los productos de capas de nubes consisten en una secuencia de descriptores de columna, cada uno asociado con un número variable de descriptores de capas de nubes. **Fuente:** [NASA Earthdata](https://www.earthdata.nasa.gov/data/catalog/larc-cloud-cal-lid-l2-01kmclay-standard-v5-00-v5-00).


## <span style='color:#ff6666'> **Requisitos**
1. Autenticación EDL (usuario/contraseña)
2. Obtener el archivo `environment.yml` e instalar el entorno conda para ejecutar el notebook.
3. `pydap>=3.5.9`


## <span style='color:#ff6666'>**Objetivos**
### Subdividir un archivo remoto

- **a)** Por variables
- **b)** Por selección espacial
- **c)** Solo datos diurnos

### Subdividir múltiples archivos remotos

- Transmitir subconjunto de datos a la estación de trabajo local

### <span style='color:#ff6666'> **Referencias**


**Getzewich, B. (2025)**. <i>CALIPSO Lidar Level 2 1 km Cloud Layer, V5-00</i> [Data set]. NASA Langley Atmospheric Science Data Center Distributed Active Archive Center. https://doi.org/10.5067/CALIOP/CALIPSO/CAL_LID_L2_01KMCLAY-STANDARD-V5-00


In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import matplotlib.pyplot as plt
import numpy as np

# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf

## Encontrar URLs OPeNDAP

### **Consultar URLs opendap usando la API CMR de NASA**

Consultamos CMR de NASA para identificar archivos remotos que intersectan la siguiente área geográfica (caja delimitadora) y cubren el siguiente rango temporal:

- -121 < longitud < -115, y 26.5 < latitud < 31
- 2 años de datos solo de primavera: del 1 de marzo al 31 de mayo (2022-2023).

Por último, nos interesan **SOLO** datos `Daytime`.


In [ ]:
Calipso_L2_ccid = "C3463063995-LARC_CLOUD" # 
bbox = [-121,26.5,-115,31] # [west, south, east, north]

# 2 years of March data
time_ranges = [[dt.datetime(year, 3, 1), dt.datetime(year, 5, 31)] for year in range(2022, 2024)]

CMR_URLs = []
args = {
    "ccid": Calipso_L2_ccid,
    "bounding_box": bbox,
    "limit": 1000,
}
cmr_urls = [url for time_range in time_ranges for url in get_cmr_urls(**args, time_range=time_range)] # you can increase the limit of results

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

## Autenticación EDL mediante earthaccess y OPeNDAP

Puedes autenticarte mediante earthaccess como se muestra abajo. Debes tener una cuenta EDL válida. Hay dos estrategias para autenticarte con `earthaccess`:

1. `strategy="interactive"`. Esto pedirá tu usuario y contraseña de EDL.
2. `strategy="netrc"`. Usa esta opción si el notebook se ejecuta en un entorno donde se puede recuperar un `.netrc` con tus credenciales.

A continuación, el valor predeterminado será `netrc`, asumiendo que el usuario ejecutó el notebook **Authenticate.ipynb**. Si no es así, puedes cambiar la estrategia a `"interactive"`.


In [ ]:
from earthaccess.exceptions import LoginStrategyUnavailable
try:
    auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials
except LoginStrategyUnavailable:
    auth = earthaccess.login(strategy="interactive", persist=True)

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()


### Acceso SOLO a metadatos con PyDAP

Podemos acceder a metadatos producidos por <span style='color:#ff6666'>**OPeNDAP**</span> para identificar las variables de interés. En particular, aquellas asociadas con valores de latitud y longitud.


In [ ]:
%%time
pyds = open_url(cmr_urls[0], protocol="dap4", session=my_session)
pyds.tree()

### Descargar variables mínimas para identificar el subconjunto espacial y datos diurnos

<font size="3"> Las coordenadas son (nombres completamente calificados):

- <font size="3"> `Latitude`
- <font size="3"> `Longitude`
- <font size="3"> `Day_Night_Flag`


<font size="3"> Antes de descargar, necesitamos identificar cualquier dimensión que también sea un arreglo del dataset.

<font size="3"> (También puede haber dimensiones con nombre, es decir, dimensiones que solo tienen nombre y que NO están

<font size="3">asociadas con datos. No necesitamos declarar esas variables.)


In [ ]:
DIMS = list(set(pyds['Latitude'].dims + pyds['Longitude'].dims + pyds['Day_Night_Flag'].dims))
dims = [dim for dim in DIMS if dim.split("/")[-1] in pyds[("/").join(DIMS[1].split('/')[:-1])].variables()] 
print("Dimensions that are also arrays: ", dims)

In [ ]:
output_path = "./data/"

### Transmitir datos


In [ ]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, 
              keep_variables = ["/Longitude", "/Day_Night_Flag"], 
              output_path=output_path)

## Inspeccionar todos los archivos descargados

<font size="3"> Aquí identificamos con más detalle el subconjunto de datos necesario en el archivo remoto, que devolverá SOLO datos dentro de nuestra caja delimitadora, para cualquier variable de interés posible.


In [ ]:
# Get data from Bounding Box
minLon, maxLon = bbox[0], bbox[2]

slices=[]
final_urls = []
for url in cmr_urls:
    filename = output_path+f"{url.split('/')[-1][:-4]}.nc4"
    dt1 = xr.open_datatree(filename).load()
    daytime_flag = dt1['Day_Night_Flag']
    # find index /data_01/longitude
    longitude = dt1['/Longitude']
    mask = (longitude >= minLon) & (longitude <= maxLon)
    idx = np.nonzero(mask.values)[0]
    daytime_flag = dt1['Day_Night_Flag'].isel(Record_Number=slice(idx[0], idx[-1]))==1
    if all(daytime_flag==0):
        final_urls.append(url)
        slices.append({"/Record_Number":(idx[0], idx[-1])})

print(f"\nOnly {len(final_urls)} out of the {len(cmr_urls)} remote files satisfy our Daylight Criteria\n")
print("Sample subsetting slices:")
slices[:4]

## Inspeccionar visualmente el rebanado para el subconjunto


In [ ]:
# Subset data
Lon = dt1['Longitude'].isel(Record_Number=slice(idx[0], idx[-1]))

# Generate masked data to visualize only

Lon_masked = xr.full_like(dt1['Longitude'], np.nan)
Lon_masked.loc[dict(
    Record_Number = Lon['Record_Number'] + idx[0]
)] = Lon

# Visualize: Plot subset of data over original data
fig, axes = plt.subplots(figsize=(10,4))
dt1['Longitude'].plot(lw=5, color='k', alpha=0.75);
Lon_masked.plot(lw=10, color="#7f00ff")
axes.set_title(r"Longitude Subset $[^\circ$E]")
plt.show()

## Descargar todos los datos de interés

<font size="3"> **PRIMERO:** necesitamos borrar todos los archivos descargados previamente para evitar colisiones de nombres.


In [ ]:
import os
import glob

fnames = [output_path+f"{fname.split('/')[-1][:-4]}.nc4" for fname in cmr_urls]
for filename in fnames:
    try:
        os.remove(filename)
    except FileNotFoundError:
        print(f"The file '{filename}' is not in there anymore")    

In [ ]:
# Will Download 34 Variables!
keep_variables = [
    '/Lidar_Surface_Detection', # <----- ALL Variables inside Group
    "/Ocean_Derived_Column_Optical_Depth", # < -- All varibles inside Group
    "/Lidar_Data_Altitudes", "/Profile_ID", "/Latitude", "/Longitude", 
    "/Profile_Time", "/Profile_UTC_Time", "/Day_Night_Flag", "/Tropopause_Height", 
    "/Tropopause_Temperature",
]


In [ ]:
%%time
dap_to_netcdf(final_urls, session=my_session,
              keep_variables = keep_variables,
              dim_slices = slices,
              output_path=output_path)

## Inspeccionar un archivo descargado (local)

<font size="3"> **NOTA:** el archivo hereda el nombre de archivo fuente mediante los metadatos OPeNDAP. Podemos recuperar el nombre de archivo fuente desde cada URL.


In [ ]:
filename = output_path+f"{final_urls[0].split('/')[-1][:-4]}.nc4"
dt1 = xr.open_datatree(filename).load()
dt1
